# Face Age Classifier
**Transfer Learning with ResNet50 & DenseNet121**

This notebook builds an age group classifier from face images using pretrained CNN models.
The dataset contains ~32K face images grouped into 3 age classes: **14-24**, **25-40**, and **41-70**.

Models used: ResNet50 (~67% val accuracy) and DenseNet121 (~70% val accuracy)

Dataset:-
- [Age Detection - Face Recognition Dataset (Kaggle)](https://www.kaggle.com/datasets/trainingdatapro/age-detection-human-faces-18-60-years)
- [Face-Age-Gender Dataset (Kaggle)](https://www.kaggle.com/datasets/aadyasingh55/face-age-gender-dataset/data)

## 1. Imports & Device Setup
Just importing everything we need and checking if there's a GPU available.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import cv2
%matplotlib inline

# PyTorch core
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# TorchVision
import torchvision.transforms.v2 as v2
from torchvision.models import resnet50, densenet121, ResNet50_Weights, DenseNet121_Weights
from torchvision.utils import make_grid

# Data utilities
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Evaluation
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 2. Configuration
All the hyperparameters in one place so it's easy to change them later.
If you're running this on Kaggle, change MAIN_PATH to your dataset path.

In [ ]:
# If running on Kaggle, update MAIN_PATH to: /kaggle/input/...
MAIN_PATH = r"C:\Users\ibrah.HIMA\OneDrive\Desktop\Full AI\Deep Learning\CNN\All Data About CNN\Face-Age-Gender Dataset\face_dataset"
TRAIN_PATH = os.path.join(MAIN_PATH, "train")
TEST_PATH  = os.path.join(MAIN_PATH, "test")

IMG_SIZE   = 224    # ResNet/DenseNet expect 224x224
EPOCHS     = 10
BATCH_SIZE = 64
PATIENCE   = 5      # early stopping patience
LR         = 1e-5   # low LR suits fine-tuning pretrained models

# ImageNet normalization values (used since we start from pretrained weights)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

NUM_WORKERS = 0

## 3. Dataset Class
This handles loading images from the DataFrame and converting the string labels
like '14-24' into integers that PyTorch can work with.

In [ ]:
class CustomDataset(Dataset):
    """
    Loads face images from a DataFrame and returns (image_tensor, label_int) pairs.

    Parameters
    ----------
    csv_file  : pd.DataFrame — must have image path as column 0, age label as column 1
    transform : torchvision transform pipeline to apply on each image
    """
    def __init__(self, csv_file, transform=None):
        # reset_index prevents iloc from breaking after train_test_split
        self.data = csv_file.reset_index(drop=True)
        self.transform = transform

        # Build a consistent string -> int mapping for class labels
        unique_labels = sorted(self.data.iloc[:, 1].unique())
        self.label2idx = {label: idx for idx, label in enumerate(unique_labels)}
        # e.g. {'14-24': 0, '25-40': 1, '41-70': 2}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_path = self.data.iloc[index, 0]
        label    = self.data.iloc[index, 1]

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, self.label2idx[label]

## 4. Load & Explore Data
Reading the CSVs and taking a first look at what we're working with.

In [ ]:
# Load the main dataset CSV (contains image paths, raw ages, gender)
train_df = pd.read_csv(os.path.join(MAIN_PATH, "train.csv"))
test_df  = pd.read_csv(os.path.join(MAIN_PATH, "test.csv"))
train_df

In [ ]:
train_df["age"].describe()

In [ ]:
# Visualize age distribution before filtering
plt.boxplot(train_df['age'])
plt.title("Age distribution (before filtering)")
plt.ylabel("Age")
plt.show()

## 5. Data Cleaning & Label Engineering
The raw ages go from 0 to 100+, so first we filter out anything outside 14-70.
Then we group them into 3 ranges instead of using exact ages.

In [ ]:
# Remove outliers: keep only ages 14 to 70
train_df = train_df[train_df['age'] <= 70][train_df['age'] >= 14]

# Visualize after filtering
plt.boxplot(train_df['age'])
plt.title("Age distribution (after filtering)")
plt.ylabel("Age")
plt.show()

In [ ]:
def set_age_range(x):
    """Map a raw integer age to one of 3 age group strings."""
    if x in range(14, 25):
        return '14-24'
    elif x in range(25, 41):
        return '25-40'
    elif x in range(41, 71):
        return '41-70'

In [ ]:
# Rename columns, apply age grouping, build absolute file paths, drop unused columns
train_df.rename(columns={'full_path': 'file'}, inplace=True)
train_df["age"]  = train_df["age"].apply(lambda x: set_age_range(x))
train_df["file"] = train_df["file"].apply(lambda x: os.path.join(MAIN_PATH, x))
train_df.drop(['id', 'gender'], axis=1, inplace=True)
train_df.reset_index(drop=True, inplace=True)
train_df

In [ ]:
# Check class distribution to spot imbalance
train_df['age'].value_counts()

In [ ]:
# Sorted class list used later for confusion matrix axis labels
labels = sorted(list(set(train_df["age"].values)))
labels

## 6. Quick look at the images
Just a helper function to display a random image so we can make sure
the paths and labels are loading correctly.

In [ ]:
def show_img():
    """Display a random image from the 14-24 age group with its label."""
    img_path = np.random.choice(train_df[train_df['age'] == '14-24']["file"].values)
    img = plt.imread(img_path)

    # Handle grayscale images (convert to RGB so the model sees 3 channels)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    plt.imshow(img)
    plt.title(f"Label: {train_df[train_df['file'] == img_path]['age'].values[0]}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

show_img()

## 7. Transforms & Data Augmentation
For training we add some augmentation (flip, rotation, color changes) to help
the model generalize. For val and test we just resize and normalize, nothing else.

In [ ]:
def get_transforms(size=IMG_SIZE, apply_on_train=False):
    """
    Build a transform pipeline for training or validation/test.

    Training pipeline includes augmentation (flip, rotation, color jitter)
    to reduce overfitting. Val/test pipeline only resizes and normalizes.

    Parameters
    ----------
    size           : int  — target image size (default 224)
    apply_on_train : bool — True to include augmentation steps

    Returns
    -------
    torchvision.transforms.v2.Compose
    """
    base = [
        v2.Resize((size, size)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ]

    augmentation = [
        v2.RandomHorizontalFlip(),
        v2.RandomRotation(20),
        v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    ]

    # ImageNet mean/std normalization (required for pretrained models)
    tail = [v2.Normalize(MEAN, STD)]

    if apply_on_train:
        return v2.Compose(base + augmentation + tail)
    return v2.Compose(base + tail)

## 8. Train / Val / Test Split
80% goes to training, 10% validation, 10% test.
We use sklearn's train_test_split instead of PyTorch's random_split
because it works directly on DataFrames.

In [ ]:
# 80% train, 10% val, 10% test
train, temp_df = train_test_split(train_df, test_size=0.2, random_state=42)
val,   test    = train_test_split(temp_df,  test_size=0.5, random_state=42)

print(f"Train  => {len(train)}")
print(f"Val    => {len(val)}")
print(f"Test   => {len(test)}")
print(f"Total  => {len(train) + len(val) + len(test)}")

In [ ]:
# Wrap DataFrames in CustomDataset; training set gets augmentation
train_ds = CustomDataset(train, get_transforms(IMG_SIZE, True))
val_ds   = CustomDataset(val,   get_transforms(IMG_SIZE))
test_ds  = CustomDataset(test,  get_transforms(IMG_SIZE))

# Verify transform pipelines
print("Train transforms:")
print(train_ds.transform)

In [ ]:
# pin_memory=False avoids a CUDA assertion bug on some Windows setups
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=False)
val_dl   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
test_dl  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

## 9. Utility Functions
denorm() reverses the ImageNet normalization so we can display images properly.
show_grid_images() is just a sanity check to see what a batch looks like after augmentation.

In [ ]:
def denorm(imgs):
    """
    Reverse ImageNet normalization so images can be displayed correctly.

    Parameters
    ----------
    imgs : torch.Tensor — normalized image batch (B, C, H, W) or single image (C, H, W)

    Returns
    -------
    torch.Tensor in [0, 1] range
    """
    mean = torch.tensor(MEAN).view(1, 3, 1, 1).to(imgs.device)
    std  = torch.tensor(STD).view(1, 3, 1, 1).to(imgs.device)

    if imgs.dim() == 3:
        imgs = imgs.unsqueeze(0)

    return imgs * std + mean

In [ ]:
def show_grid_images(dataloader):
    """Display a grid of images from one batch (useful for sanity-checking augmentation)."""
    imgs, labels = next(iter(dataloader))
    plt.figure(figsize=(30, 30))
    plt.imshow(make_grid(denorm(imgs), nrow=16).permute(1, 2, 0))
    plt.axis('off')
    plt.tight_layout()
    plt.show()

show_grid_images(train_dl)

## 10. Model Setup
We load a pretrained DenseNet121 and replace its final layer with our own head
that outputs 3 classes instead of 1000.
A small dropout is added before the linear layer to reduce overfitting.

In [ ]:
def replace_classifier(model, name):
    """
    Swap out the final classification head of a pretrained model.

    Adds a Dropout layer before the linear head to reduce overfitting.
    The number of output neurons matches the number of age classes (3).

    Parameters
    ----------
    model : pretrained PyTorch model (ResNet or DenseNet)
    name  : str — model class name, used to identify which attribute to replace

    Returns
    -------
    model with a new classification head (only this part will be trained)
    """
    NUM_CLASSES = 3  # 14-24, 25-40, 41-70

    if "DenseNet" in name:
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.classifier.in_features, NUM_CLASSES)
        )
    elif "ResNet" in name:
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.fc.in_features, NUM_CLASSES)
        )

    return model

In [ ]:
# Load pretrained DenseNet121, replace classifier, move to GPU
# Freeze is not applied here; full fine-tuning with a very low LR (1e-5) works well
Model = densenet121(weights=DenseNet121_Weights.DEFAULT)
Model = replace_classifier(Model, Model.__class__.__name__)
Model = Model.to(device)

# AdamW with weight decay is generally better than Adam for fine-tuning
optimizer = torch.optim.AdamW(Model.parameters(), lr=LR)

# Standard cross-entropy for multi-class classification
criterion = F.cross_entropy

print(f"Model: {Model.__class__.__name__}")
print(f"Classifier head: {Model.classifier}")

## 11. Training Loop
Standard training loop with early stopping. If val loss doesn't improve
for 5 epochs in a row, we stop and keep the best checkpoint.

In [ ]:
def fit(model):
    """
    Train the model with early stopping based on validation loss.

    Each epoch:
    1. Runs the training loop and collects loss + accuracy.
    2. Evaluates on the validation set.
    3. Saves the best model checkpoint when val_loss improves.
    4. Stops early if val_loss doesn't improve for PATIENCE epochs.

    Parameters
    ----------
    model : nn.Module — the model to train (uses global optimizer, criterion, device)

    Returns
    -------
    history : list of dicts with train_loss, train_acc, val_loss, val_acc per epoch
    """
    history   = []
    best_loss = float('inf')
    counter   = 0

    for epoch in range(EPOCHS):
        train_all_loss = []
        train_all_acc  = []

        model.train()

        for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            imgs, labels = batch
            imgs   = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            loss    = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            _, preds = torch.max(outputs, dim=1)
            train_all_loss.append(loss.detach().cpu().item())
            train_all_acc.append((preds == labels).float().mean().item())

        train_loss = sum(train_all_loss) / len(train_all_loss)
        train_acc  = sum(train_all_acc)  / len(train_all_acc)

        val_all_loss = []
        val_all_acc  = []

        model.eval()
        with torch.no_grad():
            for batch in val_dl:
                imgs, labels = batch
                imgs   = imgs.to(device)
                labels = labels.to(device)

                outputs = model(imgs)
                loss    = criterion(outputs, labels)

                _, preds = torch.max(outputs, dim=1)
                val_all_loss.append(loss.detach().cpu().item())
                val_all_acc.append((preds == labels).float().mean().item())

        val_loss = sum(val_all_loss) / len(val_all_loss)
        val_acc  = sum(val_all_acc)  / len(val_all_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        history.append({
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss":   val_loss,   "val_acc":   val_acc
        })

        # Save checkpoint if this is the best val_loss seen so far
        if val_loss < best_loss:
            best_loss = val_loss
            counter   = 0
            torch.save(model.state_dict(), f"best_{model.__class__.__name__}_model.pth")
            print("  >> best model saved")
        else:
            counter += 1
            print(f"  No improvement ({counter}/{PATIENCE})")
            if counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    return history

history = fit(Model)

## 12. Training Curves
Plotting loss and accuracy over epochs to see how training went.

In [ ]:
train_acc  = [x["train_acc"]  for x in history]
val_acc    = [x["val_acc"]    for x in history]
train_loss = [x["train_loss"] for x in history]
val_loss   = [x["val_loss"]   for x in history]

_, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(train_loss, label="Train",      linewidth=2)
ax[0].plot(val_loss,   label="Validation", linewidth=2, linestyle="--")
ax[0].set_title(f"{Model.__class__.__name__} Loss Over Epochs", fontsize=13, fontweight="bold")
ax[0].set_ylabel("Loss")
ax[0].set_xlabel("Epoch")
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(train_acc, label="Train",      linewidth=2)
ax[1].plot(val_acc,   label="Validation", linewidth=2, linestyle="--")
ax[1].set_title(f"{Model.__class__.__name__} Accuracy Over Epochs", fontsize=13, fontweight="bold")
ax[1].set_ylabel("Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{Model.__class__.__name__}_LOSS_ACC_Curves.png', dpi=300, bbox_inches='tight')
plt.show()

## 13. Evaluation on Test Set
Running the model on the test set and collecting all predictions
so we can look at the confusion matrix and classification report.

In [ ]:
# Collect all predictions and ground-truth labels on the test set
all_labels = []
all_preds  = []
all_imgs   = []

Model.eval()
with torch.no_grad():
    for batch in test_dl:
        imgs, lbls = batch
        imgs = imgs.to(device)
        lbls = lbls.to(device)

        outputs = Model(imgs)
        _, preds = torch.max(outputs, dim=1)

        all_imgs.extend(imgs.detach().cpu())
        all_labels.extend(lbls.detach().cpu())
        all_preds.extend(preds.detach().cpu())

In [ ]:
def Show_CMwithClassReport(all_labels, all_preds, name):
    """
    Plot the confusion matrix and print a full classification report.

    Parameters
    ----------
    all_labels : list of ground-truth integer labels
    all_preds  : list of predicted integer labels
    name       : str — model name (used in plot title and filename)
    """
    cm = confusion_matrix(np.array(all_labels), np.array(all_preds))

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'{name} Confusion Matrix', fontsize=14, fontweight='bold')
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(f'{name}_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(classification_report(all_labels, all_preds, target_names=labels))

Show_CMwithClassReport(all_labels, all_preds, Model.__class__.__name__)

## 14. Visual Predictions on Test Samples
Showing 20 random test images with their true and predicted labels.
Green means the model got it right, red means it was wrong.

In [ ]:
# Show 20 random test predictions; green title = correct, red = wrong
_, ax = plt.subplots(4, 5, figsize=(15, 12))
ax = ax.flatten()

for i in range(20):
    x = np.random.randint(0, len(all_imgs))
    img = denorm(all_imgs[x]).squeeze().permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)   # clamp to valid display range

    true_class = labels[all_labels[x]]
    pred_class = labels[all_preds[x]]
    correct    = all_labels[x] == all_preds[x]

    ax[i].imshow(img)
    ax[i].set_title(f"True: {true_class}\nPred: {pred_class}",
                    color="green" if correct else "red", fontsize=9)
    ax[i].axis("off")

plt.suptitle("Test Set Predictions (Green = Correct | Red = Wrong)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{Model.__class__.__name__}_test_predictions.png', dpi=300, bbox_inches='tight')
plt.show()